# PS-S02 – C-MAPSS Exploratory Data Analysis

This notebook provides an interactive walkthrough of the C-MAPSS dataset.
It calls the same `src/` functions used by the pipeline, so all figures match what is in `outputs/plots/`.

**Prerequisites**: Run `python run_pipeline.py --dataset all` first to generate processed files.

In [ ]:
import sys, pathlib
# Ensure the project root is on the path
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('Imports OK')

## 1. Load Dataset

In [ ]:
from src.data_loader import load_dataset, load_rul_file, load_processed_data, load_features, load_scaler

SUBSET = 'FD001'  # Change to FD002, FD003, FD004

train_raw = load_dataset(SUBSET, 'train')
test_raw  = load_dataset(SUBSET, 'test')
rul_vals  = load_rul_file(SUBSET)

print(f'\nTrain shape: {train_raw.shape}')
print(f'Test  shape: {test_raw.shape}')
print(f'RUL values: {len(rul_vals)}')
train_raw.head()

## 2. Basic Statistics

In [ ]:
print(f'Engines in training set: {train_raw["engine_id"].nunique()}')
print(f'Engines in test set:     {test_raw["engine_id"].nunique()}')

lifetimes = train_raw.groupby('engine_id')['cycle'].max()
print(f'\nEngine lifetimes:')
print(f'  Mean:   {lifetimes.mean():.1f} cycles')
print(f'  Median: {lifetimes.median():.1f} cycles')
print(f'  Min:    {lifetimes.min()} cycles')
print(f'  Max:    {lifetimes.max()} cycles')

In [ ]:
train_raw.describe().T.style.background_gradient(cmap='Blues')

## 3. Sensor Variance Analysis

In [ ]:
sensor_cols = [c for c in train_raw.columns if c.startswith('sensor_')]
variances   = train_raw[sensor_cols].var().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#e74c3c' if v <= 1e-3 else '#2ecc71' for v in variances]
ax.bar(variances.index, variances.values, color=colors, edgecolor='white')
ax.set_yscale('log')
ax.set_title(f'{SUBSET} – Sensor Variance (Red = near-constant)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sensor')
ax.set_ylabel('Variance (log scale)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Near-constant sensors (<= 0.001 variance):')
print(variances[variances <= 1e-3].index.tolist())

## 4. Sensor–Cycle Correlation (Degradation Signals)

In [ ]:
corrs = train_raw[sensor_cols + ['cycle']].corr()['cycle'].drop('cycle').sort_values()

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#e74c3c' if c < 0 else '#3498db' for c in corrs]
ax.bar(corrs.index, corrs.values, color=colors, edgecolor='white')
ax.axhline(0.1,  color='green', linestyle='--', alpha=0.7, label='|corr|=0.1 threshold')
ax.axhline(-0.1, color='green', linestyle='--', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'{SUBSET} – Sensor Correlation with Cycle (blue=increases, red=decreases)', fontsize=12, fontweight='bold')
ax.set_xlabel('Sensor')
ax.set_ylabel('Pearson Correlation')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Individual Engine Degradation

In [ ]:
# Pick most correlated sensor
best_sensor = corrs.abs().idxmax()
print(f'Best degradation sensor: {best_sensor}  (|corr|={corrs[best_sensor]:.3f})')

engines = sorted(train_raw['engine_id'].unique())[:20]
cmap    = plt.cm.plasma(np.linspace(0, 1, len(engines)))

fig, ax = plt.subplots(figsize=(14, 6))
for eng, color in zip(engines, cmap):
    d = train_raw[train_raw['engine_id'] == eng].sort_values('cycle')
    ax.plot(d['cycle'], d[best_sensor], alpha=0.5, linewidth=1, color=color)

ax.set_title(f'{SUBSET} – Engine Degradation Curves: {best_sensor}', fontsize=12, fontweight='bold')
ax.set_xlabel('Operating Cycle')
ax.set_ylabel(best_sensor)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
# Use only variable sensors
var_sensors = [c for c in sensor_cols if train_raw[c].var() > 1e-3]
corr_matrix = train_raw[var_sensors + ['cycle']].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.3, ax=ax)
ax.set_title(f'{SUBSET} – Correlation Matrix (variable sensors)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Load Processed Data (after pipeline)

In [ ]:
try:
    processed_train = load_processed_data(SUBSET, 'train')
    processed_test  = load_processed_data(SUBSET, 'test')
    print('Columns:', processed_train.columns.tolist())
    print('RUL stats (training):')
    print(processed_train[['RUL', 'RUL_capped']].describe())
except FileNotFoundError as e:
    print(f'Run the pipeline first: python run_pipeline.py --dataset {SUBSET}')

## 8. Load Feature-Engineered Data

In [ ]:
try:
    feat_train = load_features(SUBSET, 'train')
    print(f'Feature columns ({len(feat_train.columns)} total):')
    print([c for c in feat_train.columns if '_rolling_' in c or '_diff' in c][:20])
except FileNotFoundError:
    print('Run the pipeline first.')

## 9. Sensor Statistics Table

In [ ]:
try:
    stats = pd.read_csv('../outputs/statistics/sensor_statistics.csv')
    fd_stats = stats[(stats['subset'] == SUBSET) & (stats['split'] == 'train')]
    fd_stats[['sensor','mean','std','variance','corr_with_cycle','category','is_potentially_useful']]\
        .sort_values('corr_with_cycle', key=abs, ascending=False)\
        .style.background_gradient(subset=['variance'], cmap='Oranges')
except FileNotFoundError:
    print('Run the pipeline first.')